# Educational Notebook: Web Scraping Swiss Solidarity Campaign Pages

This notebook is designed for **teaching purposes**. It compares five approaches to scrape campaign information from:

- Seed page: `https://www.swiss-solidarity.org/our-work/our-past-campaigns`
- Related pages: campaign subpages discovered from the seed page

Methods showcased:
1. HTTP requests only (regex + lightweight crawling)
2. Requests + BeautifulSoup
3. Selenium (browser automation)
4. Agentic extraction with Gemini API
5. MolmoWeb-style vision/agent endpoint (minimal quick example)

The notebook intentionally starts simple and then increases capability.

## 0) Responsible Scraping Checklist

Before scraping any website:
- Check `robots.txt` and terms of use
- Use a descriptive `User-Agent`
- Add delays between requests
- Keep request volume low
- Cache results locally when possible

In this notebook we keep all examples polite and small-scale.

In [18]:
# Install only selenium in the active notebook kernel.
%pip install -q selenium

# Persist an installation log marker for reproducibility.
from pathlib import Path
Path("outputs/swiss_solidarity_scraping").mkdir(parents=True, exist_ok=True)
Path("outputs/swiss_solidarity_scraping/00_install_log.txt").write_text(
    "Executed: %pip install -q selenium\n",
    encoding="utf-8"
)

Note: you may need to restart the kernel to use updated packages.


35

In [27]:
# Core imports used throughout the notebook.
import json
import os
import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup

SEED_URL = "https://www.swiss-solidarity.org/our-work/our-past-campaigns"
BASE_DOMAIN = "https://www.swiss-solidarity.org"
HEADERS = {
    "User-Agent": "SPOC-Educational-Scraper/1.0 (teaching demo)"
}

OUTPUT_DIR = Path("outputs/swiss_solidarity_scraping")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def write_output(filename, payload):
    """Write structured outputs to JSON files in a dedicated output directory."""
    path = OUTPUT_DIR / filename
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    return str(path)


print("Seed URL:", SEED_URL)
write_output("00_config.json", {
    "seed_url": SEED_URL,
    "base_domain": BASE_DOMAIN,
    "output_dir": str(OUTPUT_DIR)
})

Seed URL: https://www.swiss-solidarity.org/our-work/our-past-campaigns


'outputs/swiss_solidarity_scraping/00_config.json'

In [28]:
# A small helper for safe, polite HTTP GET requests.
# It prints useful debug information for educational clarity.
def polite_get(url, sleep_seconds=0.6, timeout=20):
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    print(f"GET {url} -> {response.status_code} | bytes={len(response.text):,}")
    time.sleep(sleep_seconds)
    response.raise_for_status()
    return response


def extract_campaign_content(urls, max_items=8, sleep_seconds=0.25):
    """Fetch campaign pages and extract concise, meaningful content fields."""
    records = []
    for url in list(urls)[:max_items]:
        try:
            page_html = polite_get(url, sleep_seconds=sleep_seconds).text
            page_soup = BeautifulSoup(page_html, "html.parser")

            title_tag = page_soup.find("h1") or page_soup.find("title")
            title = title_tag.get_text(strip=True) if title_tag else "(no title found)"

            paragraphs = [p.get_text(" ", strip=True) for p in page_soup.find_all("p") if p.get_text(strip=True)]
            preview = " ".join(paragraphs[:2])[:320]
            body_excerpt = " ".join(paragraphs[:8])[:1200]

            records.append({
                "url": url,
                "title": title,
                "preview": preview,
                "body_excerpt": body_excerpt,
                "paragraph_count": len(paragraphs)
            })
        except Exception as e:
            records.append({
                "url": url,
                "error": str(e)
            })
    return records


write_output("01_helper_definition.json", {
    "helpers": ["polite_get", "extract_campaign_content"],
    "default_sleep_seconds": 0.6,
    "default_timeout": 20
})

'outputs/swiss_solidarity_scraping/01_helper_definition.json'

## 1) Method A: HTTP Requests Only

In this first method, we do **not** use an HTML parser.

We will:
1. Download raw HTML
2. Use regex to find `href="..."` links
3. Keep only links that look like campaign-related paths
4. Visit a few discovered subpages

This is simple and fast, but less robust than parser-based methods.

In [29]:
# Step A1: Download the seed page as raw HTML text.
seed_html = polite_get(SEED_URL).text
print("First 400 characters of HTML:\n")
print(seed_html[:400])

(OUTPUT_DIR / "02_method_a_seed_html.html").write_text(seed_html, encoding="utf-8")
write_output("02_method_a_seed_meta.json", {
    "seed_url": SEED_URL,
    "html_length": len(seed_html)
})

GET https://www.swiss-solidarity.org/our-work/our-past-campaigns -> 200 | bytes=158,480
First 400 characters of HTML:

<!DOCTYPE html>
<html lang="en">
<head>

<meta charset="utf-8">
<!-- 
	TYPO3 by Ideative

	This website is powered by TYPO3 - inspiring people to share!
	TYPO3 is a free open source Content Management Framework initially created by Kasper Skaarhoj and licensed under GNU/GPL.
	TYPO3 is copyright 1998-2026 of Kasper Skaarhoj. Extensions are copyright of their respective owners.
	Information and cont


'outputs/swiss_solidarity_scraping/02_method_a_seed_meta.json'

In [30]:
# Step A2: Extract href values using regex only (no BeautifulSoup).
# This regex captures href values enclosed in either single or double quotes.
href_pattern = re.compile(r"href=[\"']([^\"']+)[\"']", re.IGNORECASE)
raw_links = href_pattern.findall(seed_html)

# Convert relative links to absolute and keep only same-domain links.
absolute_links = [urljoin(BASE_DOMAIN, link) for link in raw_links]
same_domain_links = [
    link for link in absolute_links
    if urlparse(link).netloc == urlparse(BASE_DOMAIN).netloc
]

# Filter likely campaign-related pages.
campaign_like = sorted({
    link for link in same_domain_links
    if "/campaigns/" in link and "our-past-campaigns" not in link
})

print(f"Total hrefs found: {len(raw_links)}")
print(f"Same-domain links: {len(same_domain_links)}")
print(f"Campaign-like subpages found: {len(campaign_like)}")
print("\nSample campaign-like links:")
for link in campaign_like[:10]:
    print(" -", link)

regex_content_records = extract_campaign_content(campaign_like, max_items=6, sleep_seconds=0.2)

write_output("03_method_a_regex_links.json", {
    "raw_link_count": len(raw_links),
    "same_domain_link_count": len(same_domain_links),
    "campaign_like_links": campaign_like,
    "campaign_content_records": regex_content_records
})

Total hrefs found: 119
Same-domain links: 88
Campaign-like subpages found: 21

Sample campaign-like links:
 - https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-relief
 - https://www.swiss-solidarity.org/campaigns/coeur-a-coeur-education-is-the-best-springboard-for-the-future
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-morocco
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-turkey-and-syria
 - https://www.swiss-solidarity.org/campaigns/earthquakes-in-venezuela
 - https://www.swiss-solidarity.org/campaigns/gaza
 - https://www.swiss-solidarity.org/campaigns/heartbeats
 - https://www.swiss-solidarity.org/campaigns/humanitarian-crisis-in-sudan
GET https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland -> 200 | bytes=172,554
GET https://www.swiss-

'outputs/swiss_solidarity_scraping/03_method_a_regex_links.json'

In [31]:
# Step A3: Brute-force-ish discovery with shallow crawling.
# We crawl a few pages and collect additional campaign links using regex only.
to_visit = [SEED_URL]
visited = set()
discovered_campaigns = set(campaign_like)
max_pages = 6  # keep small for speed and politeness

while to_visit and len(visited) < max_pages:
    current = to_visit.pop(0)
    if current in visited:
        continue

    visited.add(current)
    html = polite_get(current, sleep_seconds=0.5).text
    links = [urljoin(BASE_DOMAIN, x) for x in href_pattern.findall(html)]

    for link in links:
        if urlparse(link).netloc != urlparse(BASE_DOMAIN).netloc:
            continue
        if "/campaigns/" in link and "our-past-campaigns" not in link:
            discovered_campaigns.add(link)
        # Crawl only selected campaign listing pages to avoid explosion.
        if "/campaigns/" in link and link not in visited and link not in to_visit:
            to_visit.append(link)

print(f"Visited pages: {len(visited)}")
print(f"Total campaign-related pages discovered: {len(discovered_campaigns)}")
for link in sorted(discovered_campaigns)[:15]:
    print(" -", link)

shallow_crawl_records = extract_campaign_content(sorted(discovered_campaigns), max_items=10, sleep_seconds=0.2)

write_output("04_method_a_shallow_crawl.json", {
    "visited_pages": sorted(visited),
    "discovered_campaigns": sorted(discovered_campaigns),
    "campaign_content_records": shallow_crawl_records
})

GET https://www.swiss-solidarity.org/our-work/our-past-campaigns -> 200 | bytes=158,480
GET https://www.swiss-solidarity.org/campaigns/our-current-fundraising-campaigns -> 200 | bytes=145,257
GET https://www.swiss-solidarity.org/campaigns/earthquakes-in-venezuela -> 200 | bytes=133,326
GET https://www.swiss-solidarity.org/campaigns/swiss-solidaritys-emergency-fund -> 200 | bytes=132,459
GET https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland -> 200 | bytes=103,208
GET https://www.swiss-solidarity.org/campaigns/war-in-ukraine -> 200 | bytes=214,732
Visited pages: 6
Total campaign-related pages discovered: 23
 - https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-relief
 - https://www.swiss-solidarity.org/campaigns/coeur-a-coeur-education-is-the-best-springboard-for-the-future
 - https://www.

'outputs/swiss_solidarity_scraping/04_method_a_shallow_crawl.json'

## 2) Method B: Requests + BeautifulSoup

Now we add an HTML parser (BeautifulSoup), which makes extraction cleaner and more maintainable.

We will:
1. Parse the seed page
2. Extract campaign links from anchor tags
3. Visit a few subpages
4. Extract title + short text content

In [32]:
# Step B1: Parse seed page with BeautifulSoup and collect campaign links.
soup = BeautifulSoup(seed_html, "html.parser")
soup_links = []
for a in soup.find_all("a", href=True):
    full = urljoin(BASE_DOMAIN, a["href"])
    if "/campaigns/" in full and "our-past-campaigns" not in full:
        soup_links.append(full)

campaign_links_bs4 = sorted(set(soup_links))
print(f"Campaign links (BeautifulSoup): {len(campaign_links_bs4)}")
for link in campaign_links_bs4[:10]:
    print(" -", link)

bs4_content_records = extract_campaign_content(campaign_links_bs4, max_items=8, sleep_seconds=0.2)

write_output("05_method_b_bs4_links.json", {
    "campaign_links_bs4": campaign_links_bs4,
    "campaign_content_records": bs4_content_records
})

Campaign links (BeautifulSoup): 21
 - https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-relief
 - https://www.swiss-solidarity.org/campaigns/coeur-a-coeur-education-is-the-best-springboard-for-the-future
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-morocco
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-turkey-and-syria
 - https://www.swiss-solidarity.org/campaigns/earthquakes-in-venezuela
 - https://www.swiss-solidarity.org/campaigns/gaza
 - https://www.swiss-solidarity.org/campaigns/heartbeats
 - https://www.swiss-solidarity.org/campaigns/humanitarian-crisis-in-sudan
GET https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland -> 200 | bytes=172,554
GET https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland -> 200 | bytes=

'outputs/swiss_solidarity_scraping/05_method_b_bs4_links.json'

In [33]:
# Step B2: Detect pagination on the listing page, crawl each listing page,
# and extract structured content from campaign pages.

def detect_pagination_urls(page_html, current_url):
    """Return sorted absolute URLs that look like pagination links for the current listing."""
    parsed = urlparse(current_url)
    soup_local = BeautifulSoup(page_html, "html.parser")

    candidates = {current_url}
    for a in soup_local.find_all("a", href=True):
        href = urljoin(BASE_DOMAIN, a["href"])
        p = urlparse(href)

        if p.netloc != parsed.netloc:
            continue

        same_area = p.path.startswith(parsed.path.rstrip("/")) or parsed.path.startswith(p.path.rstrip("/"))
        looks_like_page = (
            re.search(r"[?&](page|paged)=\d+", href, re.IGNORECASE)
            or re.search(r"/page/\d+/?$", p.path, re.IGNORECASE)
            or re.search(r"/\d+/?$", p.path)
        )

        if same_area and looks_like_page:
            candidates.add(href)

    return sorted(candidates)


def collect_campaign_links_from_html(html_text):
    """Extract campaign links from one HTML document."""
    soup_page = BeautifulSoup(html_text, "html.parser")
    links = set()
    for a in soup_page.find_all("a", href=True):
        full = urljoin(BASE_DOMAIN, a["href"])
        if "/campaigns/" in full and "our-past-campaigns" not in full:
            links.add(full)
    return links


def collect_campaign_links_with_selenium_buttons(seed_url, max_pages=4):
    """Handle JavaScript pagination buttons like .pagination > li:nth-child(k) > button:nth-child(1)."""
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    campaign_links = set()
    visited_pages = []

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(seed_url)
        time.sleep(2.0)

        # Page 1 content before any click.
        campaign_links |= collect_campaign_links_from_html(driver.page_source)
        visited_pages.append({"page": 1, "selector": None})

        # Click page buttons 2..max_pages using the selector shared by user.
        for page_idx in range(2, max_pages + 1):
            selector = f".pagination > li:nth-child({page_idx}) > button:nth-child(1)"
            elements = driver.find_elements(By.CSS_SELECTOR, selector)
            if not elements:
                break

            driver.execute_script("arguments[0].click();", elements[0])
            time.sleep(1.8)

            campaign_links |= collect_campaign_links_from_html(driver.page_source)
            visited_pages.append({"page": page_idx, "selector": selector})

    finally:
        driver.quit()

    return campaign_links, visited_pages


# 1) Try static pagination discovery first.
listing_page_urls = detect_pagination_urls(seed_html, SEED_URL)
pagination_mode = "static_links"
selenium_button_pages = []
all_campaign_links = set()

if len(listing_page_urls) > 1:
    print(f"Listing pages detected via static links: {len(listing_page_urls)}")
    for u in listing_page_urls[:20]:
        print(" -", u)

    for page_url in listing_page_urls:
        html = polite_get(page_url, sleep_seconds=0.4).text
        all_campaign_links |= collect_campaign_links_from_html(html)
else:
    print("Static pagination links not found. Switching to Selenium button pagination (up to 4 pages).")
    try:
        all_campaign_links, selenium_button_pages = collect_campaign_links_with_selenium_buttons(SEED_URL, max_pages=4)
        pagination_mode = "selenium_buttons"
        listing_page_urls = [f"{SEED_URL}#page={p['page']}" for p in selenium_button_pages]
        print(f"Button-pagination pages visited: {len(selenium_button_pages)}")
        for p in selenium_button_pages:
            print(" - page", p["page"], "selector:", p["selector"])
    except Exception as e:
        pagination_mode = "fallback_seed_only"
        listing_page_urls = [SEED_URL]
        all_campaign_links = collect_campaign_links_from_html(seed_html)
        print("Selenium button pagination failed, fallback to seed page only.")
        print("Reason:", e)

campaign_links_bs4 = sorted(all_campaign_links)
print(f"\nCampaign links discovered across pages: {len(campaign_links_bs4)}")
for link in campaign_links_bs4[:20]:
    print(" -", link)


# 2) Extract meaningful content from all discovered campaign pages.
records = extract_campaign_content(campaign_links_bs4, max_items=len(campaign_links_bs4), sleep_seconds=0.2)

print("\nSample extracted records:")
print(json.dumps(records[:3], indent=2, ensure_ascii=False))

write_output("06_method_b_pagination_and_records.json", {
    "pagination_mode": pagination_mode,
    "listing_page_urls": listing_page_urls,
    "selenium_button_pages": selenium_button_pages,
    "campaign_links_bs4": campaign_links_bs4,
    "records": records
})

Listing pages detected: 1
 - https://www.swiss-solidarity.org/our-work/our-past-campaigns
GET https://www.swiss-solidarity.org/our-work/our-past-campaigns -> 200 | bytes=158,480

Campaign links discovered across listing pages: 21
 - https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland
 - https://www.swiss-solidarity.org/campaigns/child-relief
 - https://www.swiss-solidarity.org/campaigns/coeur-a-coeur-education-is-the-best-springboard-for-the-future
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-morocco
 - https://www.swiss-solidarity.org/campaigns/earthquake-in-turkey-and-syria
 - https://www.swiss-solidarity.org/campaigns/earthquakes-in-venezuela
 - https://www.swiss-solidarity.org/campaigns/gaza
 - https://www.swiss-solidarity.org/campaigns/heartbeats
 - https://www.swiss-solidarity.org/campaigns/humanitarian-crisis-in-sudan
 - https://www.swiss-sol

'outputs/swiss_solidarity_scraping/06_method_b_pagination_and_records.json'

## 3) Method C: Selenium (Browser Automation)

Use Selenium when:
- Content is rendered dynamically with JavaScript
- You need to click buttons (for example pagination controls)
- You need browser-like interaction

This section demonstrates pagination detection and campaign-link extraction from rendered pages.

In [34]:
# Step C1: Selenium with pagination detection on the rendered DOM.
# Requires selenium package + Chrome + compatible chromedriver.

selenium_result = {
    "status": "not_started",
    "listing_pages": [],
    "campaign_links": [],
    "campaign_content_records": [],
    "reason": None
}

try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.common.by import By
except Exception as e:
    selenium_result["status"] = "skipped"
    selenium_result["reason"] = f"Dependencies missing: {e}"
    print("Selenium dependencies are missing in this kernel. Skipping Selenium demo.")
    print(f"Reason: {e}")
else:
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")

    def selenium_detect_listing_pages(driver, seed_url):
        """Collect pagination/listing URLs from the rendered page."""
        driver.get(seed_url)
        time.sleep(2.5)

        urls = {seed_url}
        anchors = driver.find_elements(By.CSS_SELECTOR, "a[href]")
        for a in anchors:
            href = a.get_attribute("href")
            if not href:
                continue
            if "swiss-solidarity.org" not in href:
                continue

            same_area = "/our-work/our-past-campaigns" in href
            looks_like_page = (
                re.search(r"[?&](page|paged)=\d+", href, re.IGNORECASE)
                or re.search(r"/page/\d+/?$", href)
            )
            if same_area and looks_like_page:
                urls.add(href)

        return sorted(urls)

    def selenium_collect_campaign_links(driver, listing_urls):
        links = set()
        for url in listing_urls:
            driver.get(url)
            time.sleep(1.8)

            for e in driver.find_elements(By.CSS_SELECTOR, "a[href*='/campaigns/']"):
                href = e.get_attribute("href")
                if href and "our-past-campaigns" not in href:
                    links.add(href)
        return sorted(links)

    driver = None
    try:
        driver = webdriver.Chrome(options=options)
        selenium_listing_pages = selenium_detect_listing_pages(driver, SEED_URL)
        selenium_links = selenium_collect_campaign_links(driver, selenium_listing_pages)
        selenium_content_records = extract_campaign_content(selenium_links, max_items=12, sleep_seconds=0.2)

        selenium_result["status"] = "ok"
        selenium_result["listing_pages"] = selenium_listing_pages
        selenium_result["campaign_links"] = selenium_links
        selenium_result["campaign_content_records"] = selenium_content_records

        print(f"Selenium listing pages detected: {len(selenium_listing_pages)}")
        for u in selenium_listing_pages[:10]:
            print(" -", u)

        print(f"\nSelenium campaign links discovered: {len(selenium_links)}")
        for link in selenium_links[:10]:
            print(" -", link)

        print("\nSample Selenium extracted records:")
        print(json.dumps(selenium_content_records[:3], indent=2, ensure_ascii=False))
    except Exception as e:
        selenium_result["status"] = "skipped"
        selenium_result["reason"] = f"Browser/driver execution failed: {e}"
        print("Selenium is installed, but browser/driver execution failed. Skipping Selenium run.")
        print(f"Reason: {e}")
    finally:
        if driver is not None:
            driver.quit()

write_output("07_method_c_selenium.json", selenium_result)

GET https://www.swiss-solidarity.org/campaigns/aid-in-response-to-disasters-and-crises-in-switzerland -> 200 | bytes=172,554
GET https://www.swiss-solidarity.org/campaigns/child-protection-in-switzerland -> 200 | bytes=103,208
GET https://www.swiss-solidarity.org/campaigns/child-relief -> 200 | bytes=140,721
GET https://www.swiss-solidarity.org/campaigns/coeur-a-coeur-education-is-the-best-springboard-for-the-future -> 200 | bytes=89,761
GET https://www.swiss-solidarity.org/campaigns/earthquake-in-morocco -> 200 | bytes=94,112
GET https://www.swiss-solidarity.org/campaigns/earthquake-in-turkey-and-syria -> 200 | bytes=93,595
GET https://www.swiss-solidarity.org/campaigns/earthquakes-in-venezuela -> 200 | bytes=133,326
GET https://www.swiss-solidarity.org/campaigns/gaza -> 200 | bytes=179,400
GET https://www.swiss-solidarity.org/campaigns/heartbeats -> 200 | bytes=87,947
GET https://www.swiss-solidarity.org/campaigns/humanitarian-crisis-in-sudan -> 200 | bytes=195,684
GET https://www.sw

'outputs/swiss_solidarity_scraping/07_method_c_selenium.json'

## 4) Method D: Agentic Extraction with Gemini API (Minimal)

Here, an LLM acts as an extraction agent over each listing page.

Important notes:
- Great for quick prototyping on irregular HTML
- Always validate links with deterministic code after extraction
- Keep prompts strict and request JSON-only output

To run this section, set GEMINI_API_KEY in your environment.

In [ ]:
# Step D1: Minimal Gemini extraction using paginated listing pages from Method B.
# pip package: google-genai
api_key = os.getenv("GEMINI_API_KEY")
agentic_results = {
    "status": "not_started",
    "items": [],
    "reason": None
}

if not api_key:
    agentic_results["status"] = "skipped"
    agentic_results["reason"] = "GEMINI_API_KEY not set"
    print("GEMINI_API_KEY not set. Skipping Gemini demo cell.")
else:
    from google import genai

    client = genai.Client(api_key=api_key)
    items = []

    # Keep it minimal and fast: only first 2 listing pages.
    pages_for_agent = listing_page_urls[:2] if "listing_page_urls" in globals() else [SEED_URL]

    for page_url in pages_for_agent:
        html = polite_get(page_url, sleep_seconds=0.3).text
        html_sample = html[:80000]

        prompt = (
            "Extract up to 8 campaign links from this listing HTML. "
            "Return STRICT JSON with schema: "
            "{\"campaigns\": [{\"title\": string, \"url\": string}]}. "
            "Only include absolute URLs from swiss-solidarity.org containing /campaigns/."
        )

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[prompt, html_sample],
        )

        items.append({
            "listing_page": page_url,
            "raw_model_output": response.text[:2500]
        })

    agentic_results["status"] = "ok"
    agentic_results["items"] = items
    print(json.dumps(items, indent=2, ensure_ascii=False))

write_output("08_method_d_gemini.json", agentic_results)

GEMINI_API_KEY not set. Skipping Gemini demo cell.


## 5) Method E: MolmoWeb (Minimal Fast Example)

This example assumes a MolmoWeb-style endpoint that accepts a URL and extraction instruction.

Why this is useful:
- Very fast prototyping with minimal selector code
- Can be robust to layout shifts

Because endpoint contracts vary, this uses a generic JSON request pattern.

In [ ]:
# Step E1: Minimal generic MolmoWeb request pattern over detected listing pages.
# Set MOLMOWEB_ENDPOINT, for example: http://localhost:8000/extract
molmo_endpoint = os.getenv("MOLMOWEB_ENDPOINT")
molmo_results = {
    "status": "not_started",
    "items": [],
    "reason": None
}

if not molmo_endpoint:
    molmo_results["status"] = "skipped"
    molmo_results["reason"] = "MOLMOWEB_ENDPOINT not set"
    print("MOLMOWEB_ENDPOINT not set. Skipping MolmoWeb demo cell.")
else:
    pages_for_molmo = listing_page_urls[:2] if "listing_page_urls" in globals() else [SEED_URL]
    items = []

    for page_url in pages_for_molmo:
        payload = {
            "url": page_url,
            "instruction": "Return up to 5 campaign links with title and absolute url as JSON.",
            "max_items": 5
        }

        r = requests.post(molmo_endpoint, json=payload, timeout=30)
        item = {"listing_page": page_url, "status": r.status_code}
        try:
            item["data"] = r.json()
        except Exception:
            item["data"] = r.text[:1200]
        items.append(item)

    molmo_results["status"] = "ok"
    molmo_results["items"] = items
    print(json.dumps(items, indent=2, ensure_ascii=False))

write_output("09_method_e_molmoweb.json", molmo_results)

MOLMOWEB_ENDPOINT not set. Skipping MolmoWeb demo cell.


## 6) Quick Comparison

- HTTP only: fastest to start, but brittle on complex HTML
- Requests + BeautifulSoup: strongest baseline for static pages and pagination crawling
- Selenium: best when rendered DOM or interactions are required
- Gemini agentic extraction: flexible but requires careful output validation
- MolmoWeb: rapid, instruction-driven extraction with minimal custom parsing

## 7) Suggested Student Exercises

1. Save extracted records to CSV with columns title, url, preview, source_method.
2. Compute overlap of discovered URLs between methods.
3. Add canonical URL normalization to improve deduplication.
4. Add retry logic with exponential backoff around polite_get.
5. Measure runtime per method and discuss speed vs robustness tradeoffs.